<h1> Parsing + Chunking </h1>

In [1]:
from unstructured.partition.pdf import partition_pdf #pdf files

from unstructured.chunking.title import chunk_by_title #https://docs.unstructured.io/open-source/core-functionality/chunking

from pathlib import Path
from langchain_core.documents import Document

In [3]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documentos = []
chunk_id = 1

for docs in path.iterdir():

    file_dtype = docs.suffix.lower()

    if file_dtype == ".pdf":

        parsing = partition_pdf (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing[3:], max_characters = 750)

    '''elif file_dtype == ".txt":

        parsing = partition_text (str(docs), languages = ["por"])
        chunks = chunk_elements (parsing, max_characters = 750)

    elif file_dtype == ".docx":        
        
        parsing = partition_docx (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)'''


    for chunk in chunks:

        documentos.append (
            Document (
                page_content = chunk.text,
                metadata = {
                    "title": docs.name,
                    "chunk_id": chunk_id
                }
            )
        )

        chunk_id += 1


#documentos

<hr>

<h1> Retrieval</h1>

<h3> Sparse Retrieval </h3>

In [14]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 5) #@K aqui definido

<hr>

<h3> Dense Retrieval </h3>

<h5> Embeddings & Indexing </h5>

In [25]:
from sentence_transformers import SentenceTransformer

name = "jinaai/jina-embeddings-v5-text-small-retrieval"

model = SentenceTransformer (name)

model.save (r"C:\Users\Admin\Desktop\models\Jina")

c:\Users\Admin\Desktop\ip\How To RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--jinaai--jina-embeddings-v5-text-small-retrieval. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]


In [26]:
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores #Não o melhor para produção


emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Jina")

db = FAISS.from_documents (documentos, emb_hf)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 3058.64it/s]


In [27]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 5
    }
)

<h3> Reciprocal Rank Fusion </h3>

$$
RRF (d) = \sum _{i = 1}^{n} \frac {1} {k + pos_i(d)} 
$$

In [21]:
from eval.HitRate import Reciprocal_Rank_Fusion

### EXEMPLO

query = "O que são campos físico (gravítico, eléctrico e magnético) e como é gerado um campo magnético ?"

s_retrieval = sparse_retrieval.invoke (query) #Procura Lexical
d_retrieval = dense_retrieval.invoke (query) #Procura Vetorial

'''for doc in s_retrieval:
    display (doc.metadata["chunk_id"])'''

display (d_retrieval)

for x in d_retrieval:
    print (x.metadata["chunk_id"])

#teste = Reciprocal_Rank_Fusion ([s_retrieval, d_retrieval])

'''display (teste)
print (len (teste))'''

'''x = set (y[1] for y in teste)

print (x)'''


#display (teste)
#for ordem, (doc_name, chunk_id, chunk_text, score) in enumerate (teste, start = 1):
#    print (ordem)
#    print (doc_name)
#    print (chunk_id)
#    print (chunk_text)
#    print (score)

#display (len(teste))
#display (teste)


[Document(id='7dd12589-62d5-4791-9333-c0005adff41f', metadata={'title': 'Crianças_e_jovens_(9-17)_e_Inteligência_Artificial_Generativa_em_Portugal.pdf', 'chunk_id': 353}, page_content='"Sim. Eu acho que o princípio devia ser lei, se usarmos alguma coisa com IAs um selinho, tipo, a dizer, foi feito em parte por IAs.” (Maria, 13)'),
 Document(id='aea959bd-64cf-45eb-9f64-df131d7b082a', metadata={'title': 'ABC_Máquinas_Elétricas.pdf', 'chunk_id': 123}, page_content='motociclos).'),
 Document(id='43c69a79-f9b3-4650-949c-1a158cc1504d', metadata={'title': 'TRANSFORMADORES_DE_CORRENTE_DE_BAIXA_TENSÃO_E-REDES.pdf', 'chunk_id': 405}, page_content='020 – Embalagem\n\nOs transformadores deverão ser fornecidos em embalagens que os mantenham estáveis e sem deformações.'),
 Document(id='254c2861-c72f-4f1f-8076-ab64dfcfdeb1', metadata={'title': 'ABC_Máquinas_Elétricas.pdf', 'chunk_id': 155}, page_content='[11] Propulse, http://www2.arnes.si/~ljprop1/fra22det.html, , 1997.\n\n" - do autor\n\n! - dispon

353
123
405
155
376


'x = set (y[1] for y in teste)\n\nprint (x)'

<h2> Eval Retrieval 26/06 </h2>

In [28]:
#Dataset
import json

with open (r"C:\Users\Admin\Desktop\ip\How To RAG\eval\dataset.json", "r", encoding = "utf-8") as f:
    dataset = json.load(f)

'''for dados in dataset[:10]:
    
    c = set (dados ["chunk_id"])

    x = set (y[1] for y in teste)

    i = c.intersection (x)

    print (f"Chunks Dados --> {c}")
    print (f"Chunks Retrieved --> {x}")
    print (f"Interception --> {i}")
    print ("---" * 50)'''

'''from eval.HitRate import hitrate_k_sparse_retrieval, hitrate_k_dense_retrieval, hitrate_k_hybrid_retrieval
from eval.MeanReciprocalRank import mrr_sparse_retrieval, mrr_dense_retrieval, mrr_hybrid_retrieval'''


'from eval.HitRate import hitrate_k_sparse_retrieval, hitrate_k_dense_retrieval, hitrate_k_hybrid_retrieval\nfrom eval.MeanReciprocalRank import mrr_sparse_retrieval, mrr_dense_retrieval, mrr_hybrid_retrieval'

In [29]:
from eval.Recall import recall_k_sparse_retrieval, recall_k_dense_retrieval, recall_k_hybrid_retrieval
from eval.Precision import precision_k_sparse_retrieval, precision_k_dense_retrieval, precision_k_hybrid_retrieval
from eval.HitRate import hitrate_k_sparse_retrieval, hitrate_k_dense_retrieval, hitrate_k_hybrid_retrieval
from eval.MeanReciprocalRank import mrr_sparse_retrieval, mrr_dense_retrieval, mrr_hybrid_retrieval


a = recall_k_sparse_retrieval (sparse_retrieval, dataset)
b = recall_k_dense_retrieval (dense_retrieval, dataset)
c = recall_k_hybrid_retrieval (sparse_retrieval, dense_retrieval, dataset)

d = precision_k_sparse_retrieval (sparse_retrieval, dataset)
e = precision_k_dense_retrieval (dense_retrieval, dataset)
f = precision_k_hybrid_retrieval (sparse_retrieval, dense_retrieval, dataset)

g = hitrate_k_sparse_retrieval (sparse_retrieval, dataset)
h = hitrate_k_dense_retrieval (dense_retrieval, dataset)
i = hitrate_k_hybrid_retrieval (sparse_retrieval, dense_retrieval, dataset)

j = mrr_sparse_retrieval (sparse_retrieval, dataset)
k = mrr_dense_retrieval (dense_retrieval, dataset)
l = mrr_hybrid_retrieval (sparse_retrieval, dense_retrieval, dataset)




print (f"{a}\n{b}\n{c}")
print ("---" * 50)

print (f"{d}\n{e}\n{f}")
print ("---" * 50)

print (f"{g}\n{h}\n{i}")
print ("---" * 50)

print (f"{j}\n{k}\n{l}")
print ("---" * 50)




{'Recall@K Sparse Retrieval:': 0.41778711484593833}
{'Recall@K Dense Retrieval:': 0.5110702614379086}
{'Recall@K Hybrid Retrieval:': 0.574935807656396}
------------------------------------------------------------------------------------------------------------------------------------------------------
{'Precision@K Sparse Retrieval:': 0.21764705882352928}
{'Precision@K Dense Retrieval:': 0.28235294117647036}
{'Precision@K Hybrid Retrieval:': 0.2136437908496731}
------------------------------------------------------------------------------------------------------------------------------------------------------
{'HitRate@K Sparse Retrieval:': 0.7058823529411765}
{'HitRate@K Dense Retrieval:': 0.8235294117647058}
{'HitRate@K Hybrid Retrieval:': 0.8970588235294118}
------------------------------------------------------------------------------------------------------------------------------------------------------
{'Mean Reciprocal Rank Sparse Retrieval': 0.5549019607843138}
{'Mean Reciproc

<h3> HitRate@5 </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 5)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 5
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

<h3> HitRate@10 </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 10)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

<h3> HitRate@15 </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 15)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 15
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

<h3> MRR </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 15)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 15
    }
)

x = mrr_sparse_retrieval (sparse_retrieval_1, dataset)
y = mrr_dense_retrieval (dense_retrieval_1, dataset)
a = mrr_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (a)

<hr>

<h6> código solto </h6>

In [ ]:
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores #Não o melhor para produção
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface

In [ ]:
emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (documentos, emb_hf)

In [ ]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

In [ ]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 10)

In [ ]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 20
    }
)

#x = dense_retrieval.invoke ("O que é a corrente dinâmica estipulada (Idyn)?")

In [ ]:
### EXEMPLO

query = "Em que consiste o relatório da rede EU Kids Online ?"

s_retrieval = sparse_retrieval.invoke (query)
d_retrieval = dense_retrieval.invoke (query)

teste = Reciprocal_Rank_Fusion ([s_retrieval, d_retrieval])

for ordem, (doc_name, chunk_id, chunk_text, score) in enumerate (teste, start = 1):
    print (ordem)
    print (doc_name)
    print (chunk_id)
    print (chunk_text)
    print (score)

#display (teste)


<h1> Reranking </h1>

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder_model = AutoModelForSequenceClassification.from_pretrained (path, device_map = "cuda")
cross_encoder_model_tokenization = AutoTokenizer.from_pretrained (path)

<h4> Cross Encoder </h4>

In [ ]:

query_reranker = [query] * len (teste)
chunks = [t[2] for t in teste]
docs_names = [t[0] for t in teste]
chunks_ids = [t[1] for t in teste]

pares = list(zip(query_reranker, chunks)) # [query, chunks]
#print (pares)

inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True)
#print (inputs)

cross_encoder_model.eval()
with torch.no_grad():
    logits = cross_encoder_model (**inputs).logits
    #print (logits)

#Organiza os logits com os chunks correspondentes #Muita Atenção! Para calcular HitRate e MRR com as funções criadas é necessário estar no mesmo formato do output do RRF.
rerank = sorted(zip(docs_names, chunks_ids, chunks, logits.tolist()), key = lambda x: x[3], reverse = True) #x[3] define que arg é usado pelo reverse

#display (rerank)


<h4> Ranking </h4>

In [ ]:
final = [(docs, chunks, chunks_content, scores) for docs, chunks, chunks_content, scores in rerank [:5]]

#display (final[1])

<h3> Eval RAG + Reranking </h3>

In [ ]:
from eval.MeanReciprocalRank import mrr_ranker_sparse_system, mrr_ranker_dense_system, mrr_ranker_hybrid_system

f = mrr_ranker_sparse_system (sparse_retrieval, dataset)
h = mrr_ranker_dense_system (dense_retrieval, dataset)
i = mrr_ranker_hybrid_system (sparse_retrieval, dense_retrieval, dataset)

display (f)
display (h)
display (i)


<hr>

<h1> Eval Pré LLM </h1>

In [ ]:
#Cross Encoder Model para Rerank
path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder_model = AutoModelForSequenceClassification.from_pretrained (path)
cross_encoder_model_tokenization = AutoTokenizer.from_pretrained (path)

#Definição de Sparse Retrieval
sparse_retrieval_pre_llm = BM25Retriever.from_documents (documentos, k = 15) #@K aqui definido

#Definição de Dense Retrieval
dense_retrieval_pre_llm = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 15
    }
)

dados = []

for data in dataset:

    q = data["query"]

    #print (q)
    sp_retr = sparse_retrieval_pre_llm.invoke (q)
    den_retr = dense_retrieval_pre_llm.invoke (q)
    
    rrf = Reciprocal_Rank_Fusion ([sp_retr, den_retr])

    query_reranker = [q] * len (rrf)
    chunks = [t[2] for t in rrf]
    docs_names = [t[0] for t in rrf]
    chunks_ids = [t[1] for t in rrf]

    pares = list(zip(query_reranker, chunks)) # [query, chunks]
    #print (pares)

    inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True)
    #print (inputs)

    cross_encoder_model.eval()
    with torch.no_grad():
        logits = cross_encoder_model (**inputs).logits
        #print (logits)

    #Organiza os logits com os chunks correspondentes #Muita Atenção! Para calcular HitRate e MRR com as funções criadas é necessário estar no mesmo formato do output do RRF.
    rerank = sorted(zip(chunks, logits.tolist()), key = lambda x: x[1], reverse = True) #x[3] define que arg é usado pelo reverse

    rank = [(chunks, scores) for chunks, scores in rerank [:5]]

    dados.append (q)
    dados.append (rank)


In [ ]:
dados

<h4> Context Recall </h4>
<p> Dos “aspectos necessários para responder corretamente à pergunta”, quantos estão presentes no contexto final? </p>

<p> “Faltou alguma coisa importante no contexto?” <b> Mede omissões </b> </p>

In [ ]:
results = [4/5, 3/5, 3/5, 1/5, 1/5, 1/5, 1/5, 2/5, 1/5, 2/5, 1/5, 2/5, 2/5, 1/5, 2/5, 1/5, 1/5, 1/5, 2/5, 0, 1/5, 3/5, 2/5, 1/5, 1/5, 2/5, 2/5, 1/5, 2/5]

context_recall_final = sum(results) / len(results)
 
print (sum(results) / len(results))

<h4> Context Precision </h4>
<p> Dos chunks que foram incluídos, quantos são realmente necessários para responder à pergunta? </p>

<p> “Há coisas inúteis no contexto?” <b> Mede ruído </b></p>

In [ ]:
results = [3/5, 2/5, 3/5, 1/5, 1/5, 1/5, 1/5, 2/5, 1/5, 2/5, 1/5, 2/5, 2/5, 1/5, 2/5, 1/5, 1/5, 1/5, 2/5, 0, 1/5, 3/5, 2/5, 1/5, 1/5, 2/5, 2/5, 1/5, 2/5]

context_precision_final = sum(results) / len(results)

print (sum(results) / len(results)) 

<hr>

<hr>

<h3> LLM </h3>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

path = r"C:\Users\Admin\Desktop\models\Phi-3.5-mini-instruct-Q4"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer2 = AutoTokenizer.from_pretrained (path)

device


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print (device)
print (torch.__version__)
print (torch.version.cuda)

<h1> Answer Relevance </h1>

In [ ]:
with open (r"C:\Users\Admin\Desktop\ip\How To RAG\eval\dataset_pós_answer.json", "r", encoding = "utf-8") as f:
    dataset_fim = json.load(f)

In [ ]:
#print (len(dataset_fim))
display (dataset_fim[0])

In [ ]:
dataa = []

for q in dataset_fim:
    #print (q)

    #Selecionar query
    query = q["query"]

    sparse_ret = sparse_retrieval.invoke (query) #Sparse Retrieval
    dense_ret = dense_retrieval.invoke (query) #Dense Retrieval

    rrf = Reciprocal_Rank_Fusion ([sparse_ret, dense_ret]) #Reciprocal Rank Fusion

    #Preparar o input para o Cross Encoder Model que precisa de receber [query, chunk]
    query_reranker = [query] * len (rrf) 
    chunks = [t[2] for t in rrf]

    pares = list(zip(query_reranker, chunks))
    ####

    inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True).to(device) #Tokenization input

    cross_encoder_model.eval()
    with torch.no_grad():
        logits = cross_encoder_model (**inputs).logits #logits

    rerank = sorted(zip(chunks, logits.tolist()), key = lambda x: x[1], reverse = True) #Organização do output
    #display (rerank)

    feed_llm = [chunks for chunks, scores in rerank [:5]] #Os 5 melhores..
    #display (feed_llm)

    
    prompt = query

    ##Analisar isto ##Interssante perceber se tokens especiais influenciam a resposta do modelo. ##Prompt Engineering https://www.youtube.com/watch?v=ysPbXH0LpIE&t=958s
    chat_template = f"""
    <|system|>
    És um assistente de IA alimentado por um sistema RAG, que tem como único e principal objetivo responder a perguntas realizadas pelos utilizadores de acordo com o contexto fornecido.

    Deves responder de maneira curta e direta, num tom amigável.
    
    Regras:
        1. É fulcral e obrigatório que respondas apenas consoante o contexto fornecido.
        2. Não uses conhecimento externo.
        3. Se a resposta não existir no contexto, responde exatamente: "Informação não presente no contexto fornecido"
        4. Responde de forma breve e direta.
        5. Formata a resposta de forma clara.

    Tens como principal e única função responder a perguntas de acordo com o Contexto fornecido. Não deves inventar nem aceder a informação externa. 
    Fazes parte de um sistemas RAG e deves apenas responder de acordo com o Contexto fornecido. <|end|>

    <|user|>
    <Contexto>

    {feed_llm}
    
    </Contexto>
    
    Pergunta:
    {prompt} <|end|>

    <|assistant|>

    """
    #print (chat_template)

    
    model_inputs = tokenizer2 (chat_template, return_tensors = "pt").to(device) #Tokenização 

    with torch.no_grad():
        generated_ids = model.generate(**model_inputs, max_new_tokens = 512) #Inferência

    ## Recortar a pergunta da resposta
    input_length = model_inputs["input_ids"].shape[1]
    response_tokens = generated_ids[0][input_length:]
    ##

    output = tokenizer2.decode(response_tokens, skip_special_tokens = True) ##Resposta final
    #print (output)

    dataa.append (query)
    dataa.append (feed_llm)
    dataa.append (output)
    

In [ ]:
torch.cuda.empty_cache()

In [ ]:
import pandas as pd

df = pd.DataFrame (dataa, columns = ["x"])

df.to_csv ("Phi-3.5-mini-instruct-Q4 - Output Analysis.csv", index = False)
#dataa

<h3> Answer Relevance </h3>

<p> Avalia até que ponto a resposta responde à pergunta do utilizador. </p>
<p> Ajuda do GPT </p>
<p> Qwen2.5-0.5B-Instruct </p>

In [ ]:
answer_rel = [1, 0.5, 1, 0.6, 0.2, 0.6, 0.4, 1, 0.4, 1, 1, 0.9, 1, 1, 1, 0.2, 1, 0.8, 0.7, 0.2, 0, 0.9, 0, 0.15, 0.70, 0.90, 0.75]

answer_rel_final = sum(answer_rel) / len(answer_rel)

print (len(answer_rel))
print (sum(answer_rel) / len(answer_rel))

<h3> Faithfulness </h3>
<p> Avalia se a resposta está suportada pelos documentos recuperados pelo sistema RAG. </p>
<p> Explicar que os modelos podem inventar informação e isso tem que ter menos pontos. </p>
<p> Qwen2.5-0.5B-Instruct </p>

In [ ]:
faith = [1, 0.1, 1, 0.2, 0, 0.3, 0.1, 1, 0.2, 1, 1, 0.8, 1, 1, 0.9, 0.1, 1, 0.9, 0.8, 0, 0, 0.9, 0, 0.85, 0.35, 0.55, 0.40]

faith_final = sum(faith) / len(faith) 

print (len(faith))
print (sum(faith) / len(faith))

<hr>

<h1> RAGAS Score de um Sistema RAG </h1>

<p> Hybrid Retrieval -> Reranker -> Modelo (Qwen2.5-0.5B-Instruct) </p>

<h1> SUSPEITO </h1>

In [ ]:
retr_score = (z["HitRate@K Chunk Hybrid Retrieval"] * 0.40) + (a["Mean Reciprocal Rank Chunks Hybrid Retrieval"] * 0.60)
context_score = (context_recall_final * 0.5) + (context_precision_final * 0.5)
gen_score = (answer_rel_final * 0.5) + (faith_final * 0.5)

print (retr_score)
print (context_score)
print (gen_score)


#print (gen_score)

#display (hitrate)
#display (mrr_hybrid)
#display (mrr_ranker_hybrid)

RAGAS = (retr_score * 0.35) + (context_score * 0.35) + (gen_score * 0.30)

print (f"{RAGAS * 100}")


<hr>

<hr>

In [ ]:
'''from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained ("mistralai/Mistral-7B-Instruct-v0.3")
tokenization = AutoTokenizer.from_pretrained ("mistralai/Mistral-7B-Instruct-v0.3")

model.save_pretrained (r"C:\Users\Admin\Desktop\models\Mistral-7B-Instruct-v0.3")
tokenization.save_pretrained (r"C:\Users\Admin\Desktop\models\Mistral-7B-Instruct-v0.3")
'''

<h1> Answer Relevance </h1>
<p> Phi-3.5-mini-instruct-Q4 </p>

In [ ]:
answer_rel = [1, 1, 0.70, 0.95, 1, 1, 0.4, 0.2, 0.95, 0.5, 1, 0.95, 1, 1, 0.95, 0.8, 0.85, 0.9, 0.3, 0.9, 0.95, 1, 0.4, 0.2, 0.9, 1, 0.98]

answer_rel_final = sum(answer_rel) / len(answer_rel)

answer_rel_final

<h1> Faithfulness </h1>

<p> Phi-3.5-mini-instruct-Q4 </p>

In [ ]:
faith = [1, 1, 1, 1, 1, 1, 0.6, 0, 0.95, 1, 1, 0.95, 1, 1, 1, 1, 1, 0.95, 0.2, 0.9, 0.85, 0.95, 0.2, 0.1, 0.75, 0.9, 0.97]

faith_final = sum(faith) / len(faith)

faith_final

<h1> RAGAS Score de um Sistema RAG </h1>

<p> Hybrid Retrieval -> Reranker -> Modelo (microsoft/Phi-3.5-mini-instruct) </p>

In [ ]:
gen_score_2 = (answer_rel_final * 0.5) + (faith_final * 0.5)

print (gen_score_2)
#print (gen_score_2)

RAGAS_2 = (retr_score * 0.35) + (context_score * 0.35) + (gen_score_2 * 0.30)

display (RAGAS_2 * 100)